In [ ]:
from __future__ import annotations
from typing import TypedDict, List, Annotated
from pydantic import BaseModel, Field
from langgraph.types import Send
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph import StateGraph,START,END
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI 
from dotenv import load_dotenv
from typing import TypedDict , Literal , Annotated
import os
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()
import operator
from uuid import uuid4

In [ ]:
class Task(BaseModel):
    id:  str = Field(default_factory=lambda: str(uuid4()))
    title: str
    brief: str = Field(..., description="What to cover")

In [ ]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [ ]:
class State(TypedDict):
    topic: str
    plan: Plan
    # reducer: results from workers get concatenated automatically
    sections: Annotated[List[str], operator.add]
    final: str

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.5)

In [ ]:
def orchestrator(state: State) -> dict:

    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic."
                )
            ),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan": plan}

In [ ]:
def fanout(state: State):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks] 

In [ ]:
def worker(payload: dict) -> dict:

    # payload contains what we sent
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = llm.invoke(
        [
            SystemMessage(content="Write one clean Markdown section."),
            HumanMessage(
                content=(
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Brief: {task.brief}\n\n"
                    "Return only the section content in Markdown."
                )
            ),
        ]
    ).content.strip()

    return {"sections": [section_md]}

In [ ]:
# from pathlib import Path

# def reducer(state: State) -> dict:
    
#     title = state["plan"].blog_title
#     body = "\n\n".join(state["sections"]).strip()

#     final_md = f"# {title}\n\n{body}\n"

#     # ---- save to file ----
#     filename = title.lower().replace(" ", "_") + ".md"
#     output_path = Path(filename)
#     output_path.write_text(final_md, encoding="utf-8")

#     return {"final": final_md}


from pathlib import Path

def reducer(state: State) -> dict:

    title = state["plan"].blog_title

    print("=" * 60)
    print("Reducer called")
    print("Sections:", len(state["sections"]))
    print(state["sections"])
    print("=" * 60)

    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{body}\n"

    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)

    print("Writing to:", output_path.resolve())
    print("Characters:", len(final_md))

    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md}

In [ ]:
graph = StateGraph(State)
graph.add_node("orchestrator", orchestrator)
graph.add_node("worker", worker)
graph.add_node("reducer", reducer)
graph.add_edge(START,"orchestrator")
graph.add_conditional_edges("orchestrator",fanout,["worker"])
graph.add_edge("worker","reducer")
graph.add_edge("reducer",END)

workflow = graph.compile()

In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
out = workflow.invoke({"topic": "Write a blog on Self Attention", "sections": []})